# Probeprüfung AP01: Management & Nutzung relationaler Daten (Probeprüfung – Musterlösung)

Aufgebaut wie die Prüfung *AP01* — gleiche Teile, gleiche Punkte, gleiches Bewertungsschema —, aber mit der
fiktiven Datenbank der Fitnesskette **MoveFit** statt `cinema.db`.

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

| Teil | Thema | Aufgaben | Punkte |
|---|---|---|---|
| A | Konzeptfragen: Datenmodellierung & Design | A1–A3 | 12 |
| B | SQL-Fehleranalyse (fitness.db) | B1–B2 | 16 |
| C | SQL-Abfragen schreiben (fitness.db) | C1–C3 | 12 |
| **Total** | | **8 Aufgaben** | **40** |

### Bewertungsschema

| Kategorie | Beschreibung | Bewertung | 4 Pkt | 8 Pkt |
|---|---|---|---|---|
| Korrekt | Vollständig und fachlich einwandfrei; SQL liefert das richtige Ergebnis | 100% | 4 | 8 |
| Minimale Mängel | Kleine Fehler (z. B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau | 75% | 3 | 6 |
| Mittlere Mängel | Grössere Fehler (z. B. fehlende Spalten, falsche Filterung) oder Begründung fehlt | 50% | 2 | 4 |
| Gravierende Mängel | Ansatz im Kern falsch (z. B. falscher JOIN-Typ, fehlende Tabelle) | 25% | 1 | 2 |
| Nicht lauffähig | Antwort fehlt oder Code nicht lauffähig | 0% | 0 | 0 |

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege gelten als korrekt.

*Empfohlene Bearbeitungszeit: 60 Minuten. Arbeite «open book» (Factsheet, Referenz, eigene Übungen), aber ohne KI —
wie in der Prüfung.*

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup aufgebaut und getestet.

**Backup:** `Data/fitness.db` (SQLite) — wird automatisch verwendet, falls PostgreSQL nicht erreichbar ist.

**Tabellen:** `studios`, `courses`, `trainers`, `sessions`, `members`, `bookings` — Beschreibung und Datenmodell in
`Data/fitness_beschreibung.md`.

**Antworten auf Konzeptfragen** tragen Sie direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [ ]:
# Libraries
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

In [ ]:
# Zugangsdaten (Werte aus docker-compose.yml, über Umgebungsvariablen überschreibbar)
user     = os.getenv('PG_USER', 'pgadmin')
password = os.getenv('PG_PASSWORT', 'geheim')
port     = os.getenv('PG_PORT', '5432')
database = os.getenv('PG_DATABASE', 'postgres')

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in (os.getenv('PG_HOST'), 'db', 'localhost'):
    if not candidate:
        continue
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

if host:
    db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'
else:
    # Backup: SQLite-Datei verwenden (gleiche Tabellen und Inhalte)
    print('Keine PostgreSQL-Verbindung gefunden -> verwende das SQLite-Backup Data/fitness.db')
    db_connection_url = 'sqlite:///Data/fitness.db'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Verbindung:', engine.dialect.name, '| Host:', host)
engine.dispose()

In [ ]:
# Tabellen aus der SQLite-Datei fitness.db nach PostgreSQL übertragen
tables = ['studios', 'courses', 'trainers', 'sessions', 'members', 'bookings']

if db_connection_url.startswith('postgresql'):
    sqlite_engine = create_engine('sqlite:///Data/fitness.db')
    engine = create_engine(db_connection_url)
    for table in tables:
        df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
        df.to_sql(table, engine, if_exists='replace', index=False)
        print(f'{table:<10}{len(df):>6} Zeilen')
    sqlite_engine.dispose()
    engine.dispose()

---
**▶ Ab hier beginnt die bewertete Probeprüfung — 40 Punkte**

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `fitness.db` werden nicht gewertet.*

### A1 (4 Punkte)

Ein Kollege schlägt vor, die Spalten `studio_name` und `studio_city` direkt in die Tabelle `sessions` aufzunehmen:
*«Dann sparen wir uns bei jeder Auswertung den JOIN mit `studios`.»*

Ist das eine gute Idee? Begründen Sie Ihre Antwort mit einem konkreten Szenario aus dem Betrieb der Fitnesskette.

**Ihre Antwort:**

Nein. Name und Stadt eines Studios hängen nur von `studioid` ab und stehen bereits einmal in `studios`. In `sessions`
würden sie bei **jedem** der 300 Kurstermine wiederholt (z. B. ca. 75-mal «MoveFit Oerlikon»). Das ist Redundanz und
verletzt die Normalisierung. Szenario: Das Studio «MoveFit Bahnhof» wird in «MoveFit Bern Bahnhof» umbenannt. Dann
müssten alle zugehörigen Termine angepasst werden. Wird ein Teil vergessen, existieren zwei Namen für dasselbe Studio
(Update-Anomalie, inkonsistente Auswertungen). Ausserdem könnte ein neues Studio ohne Termine (`ST06` Wankdorf) gar
nicht erfasst werden (Einfüge-Anomalie). Der JOIN über den Fremdschlüssel `studioid` ist der korrekte Weg.

### A2 (4 Punkte)

Das Datenteam analysiert die Spalte `members.email` und findet unter anderem folgende Werte:

```
anna.mueller@example.ch, NULL (15-mal), '' (leerer Text), 'n/a', 'jonas.huber@example'
```

Beschreiben Sie die konkreten Datenprobleme und nennen Sie die passenden Data-Quality-Eigenschaften aus den
Vorlesungsfolien (z. B. accurate, consistent, complete, reliable).

**Ihre Antwort:**

Es gibt zwei Arten von Problemen:
1. **Fehlende Werte:** `NULL`, der leere Text `''` und der Platzhalter `'n/a'` bedeuten alle «keine E-Mail bekannt».
   Verletzt ist **complete**. Zusätzlich ist «fehlt» auf drei verschiedene Arten dargestellt, das verletzt
   **consistent** (eine Abfrage `WHERE email IS NULL` findet nur einen Teil).
2. **Formal ungültiger Wert:** `jonas.huber@example` hat keine Top-Level-Domain (z. B. `.ch`). Der Wert ist vorhanden,
   aber falsch, die E-Mail ist nicht zustellbar. Verletzt ist **accurate**.

### A3 (4 Punkte)

Der Kurstermin `SE0005` hat eine Kapazität von 12 Plätzen, in `bookings` gibt es dafür aber 13 nicht stornierte
Buchungen. Eine Kollegin sagt: *«Das ist ein Datenfehler, die 13. Buchung muss gelöscht werden.»*
Ist das **zwingend** ein Datenfehler? Begründen Sie in einem Satz.

**Ihre Antwort:**

Nein, nicht zwingend: Eine bewusste Überbuchung (weil erfahrungsgemäss Mitglieder als `no-show` nicht erscheinen)
oder ein vom Trainer zusätzlich zugelassener Platz ist geschäftlich denkbar — der Fall weicht aber von der Regel
«Buchungen ≤ Kapazität» ab und muss gegen die Buchungsregeln geprüft werden, statt die Buchung ungeprüft zu löschen.

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis.*

### B1 (8 Punkte)

**Ziel:** Anzahl Buchungen pro Kurs (`title`) — für **alle** Kurse in der Datenbank.

```sql
SELECT c.title, COUNT(*) AS booking_count
FROM sessions s
JOIN bookings b ON s.sessionid = b.sessionid
JOIN courses c  ON s.courseid  = c.courseid
GROUP BY c.title;
```

Die Tabelle `courses` enthält **15 Kurse**. Tatsächliches Ergebnis der SQL-Abfrage: nur **14 Zeilen**.

Die Leiterin Kursplanung fragt: *«Zeig mir auch Kurse, die noch nie gebucht wurden.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 15 Kurse** erscheinen und ein Kurs ohne Buchungen
korrekt mit `0` ausgewiesen wird. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

Die `INNER JOIN`s liefern nur Kurse, die mindestens einen Termin **und** eine Buchung haben. Der Kurs «Aqua Fit»
(`CO15`) wurde noch nie angeboten und fällt deshalb weg.
1. Die Abfrage muss von `courses` ausgehen, und beide Joins werden zu `LEFT JOIN`
   (`courses → sessions → bookings`). So bleiben alle 15 Kurse erhalten.
2. `COUNT(*)` → `COUNT(b.bookingid)`: `COUNT(*)` würde die durch den `LEFT JOIN` erzeugte NULL-Zeile von «Aqua Fit»
   als 1 zählen, `COUNT(b.bookingid)` ignoriert NULL und liefert korrekt 0.

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
SELECT
    c.title,
    COUNT(b.bookingid) AS booking_count
FROM courses c
LEFT JOIN sessions s ON c.courseid  = s.courseid
LEFT JOIN bookings b ON s.sessionid = b.sessionid
GROUP BY c.title
ORDER BY booking_count DESC
""", con=engine)

engine.dispose()
df_B1

In [ ]:
# Kontrolle der Musterlösung
assert len(df_B1) == 15 and dict(zip(df_B1['title'], df_B1['booking_count'].astype(int)))['Aqua Fit'] == 0

### B2 (8 Punkte)

**Ziel:** Tatsächlich erzielter Umsatz pro Studio (`name`). Stornierte Buchungen (`status = 'cancelled'`) werden
vollständig zurückerstattet; Buchungen mit `no-show` bleiben bezahlt.

```sql
SELECT st.name, SUM(s.base_price) AS revenue
FROM studios st
JOIN sessions s ON st.studioid  = s.studioid
JOIN bookings b ON s.sessionid  = b.sessionid
GROUP BY st.name
ORDER BY revenue DESC;
```

Tatsächliches Ergebnis (Ausschnitt):

```
               name   revenue
0  MoveFit Oerlikon   14523.0
1   MoveFit Bahnhof   10548.0
...
```

Die Finanzabteilung meldet, dass die Bankeingänge deutlich tiefer liegen.

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage (Umsatz auf 2 Dezimalstellen gerundet). Begründen Sie jede
Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

Die Abfrage hat zwei fachliche Fehler:
1. `SUM(s.base_price)` summiert den **Listenpreis** des Termins für jede Buchung. Rabatte (Student, Senior, Premium,
   10er-Abo, Gutscheine) werden ignoriert, der Umsatz ist zu hoch. Richtig ist der tatsächlich bezahlte Betrag
   `SUM(b.paid_price)`.
2. Stornierte Buchungen werden mitgezählt, obwohl sie zurückerstattet werden. Ergänzt wird
   `WHERE b.status <> 'cancelled'`. `no-show` bleibt drin, weil diese Buchungen bezahlt sind.

Zusätzlich wird mit `ROUND(CAST(... AS NUMERIC), 2)` auf Rappen gerundet (in PostgreSQL ist das `CAST` nötig).
Der `INNER JOIN` ist hier korrekt: Studios ohne Buchungen haben keinen Umsatz.

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
SELECT
    st.name,
    ROUND(CAST(SUM(b.paid_price) AS NUMERIC), 2) AS revenue
FROM studios st
JOIN sessions s ON st.studioid = s.studioid
JOIN bookings b ON s.sessionid = b.sessionid
WHERE b.status <> 'cancelled'
GROUP BY st.name
ORDER BY revenue DESC
""", con=engine)

engine.dispose()
df_B2

In [ ]:
# Kontrolle der Musterlösung
assert abs(float(df_B2['revenue'][0]) - 11322.40) < 0.01 and len(df_B2) == 5

---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `fitness.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen.

### C1 (4 Punkte)

Schreiben Sie eine SQL-Abfrage, die alle Mitglieder (`memberid`, `firstname`, `lastname`, `email`) ausgibt, deren
E-Mail **fehlt oder ungültig** ist. Als ungültig gilt eine E-Mail, die leer ist, kein `@` enthält oder nach dem `@`
keinen Punkt hat. Sortieren Sie nach `memberid`. Begründen Sie Ihre Filterbedingung — insbesondere den Umgang mit NULL.

**Begründung der Vorgehensweise:**

Alle Spalten liegen in `members`, ein JOIN ist nicht nötig. `email NOT LIKE '%@%.%'` erfasst leere Texte, Werte ohne
`@` (`n/a`) und Werte ohne Punkt nach dem `@`. NULL-Werte erfasst die Bedingung **nicht**: Ein Vergleich mit NULL
ergibt UNKNOWN, die Zeile fällt aus dem WHERE heraus. Deshalb braucht es zusätzlich `email IS NULL`, verknüpft mit `OR`.

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
SELECT
    memberid,
    firstname,
    lastname,
    email
FROM members
WHERE email IS NULL
   OR email NOT LIKE '%@%.%'
ORDER BY memberid
""", con=engine)

engine.dispose()
df_C1

In [ ]:
# Kontrolle der Musterlösung
assert len(df_C1) == 18

### C2 (4 Punkte)

Wie hoch ist der durchschnittliche bezahlte Preis (`paid_price`) pro Kurskategorie (`category`)? Geben Sie `category`
und den Durchschnittspreis (`avg_price`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`.
Sortieren Sie absteigend nach `avg_price`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

`category` liegt in `courses`, `paid_price` in `bookings`. Die beiden Tabellen sind nicht direkt verbunden — der Weg
führt über `sessions` (`courses.courseid = sessions.courseid`, `sessions.sessionid = bookings.sessionid`).
`GROUP BY c.category` gruppiert die Buchungen pro Kategorie, `AVG(b.paid_price)` bildet den Durchschnitt. Ein
`INNER JOIN` ist korrekt: Kurse ohne Buchungen haben keinen bezahlten Preis und dürfen den Durchschnitt nicht verfälschen.

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
SELECT
    c.category,
    ROUND(CAST(AVG(b.paid_price) AS NUMERIC), 2) AS avg_price
FROM courses c
JOIN sessions s ON c.courseid  = s.courseid
JOIN bookings b ON s.sessionid = b.sessionid
GROUP BY c.category
ORDER BY avg_price DESC
""", con=engine)

engine.dispose()
df_C2

In [ ]:
# Kontrolle der Musterlösung
assert df_C2['category'].tolist() == ['Pilates', 'Yoga', 'Kraft', 'Ausdauer', 'Gesundheit', 'Tanz']
assert float(df_C2['avg_price'][0]) == 24.86

### C3 (4 Punkte)

Welche Trainer haben **mehr als 15** Kurstermine geleitet? Geben Sie den vollen Namen als `trainer_name`
(Vorname Leerschlag Nachname) und die Anzahl Termine als `session_count` aus, absteigend nach `session_count` und bei
Gleichstand nach `trainer_name`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

Es braucht nur `trainers` (Name) und `sessions` (wer welchen Termin leitet), verbunden über `trainerid`.
`GROUP BY` fasst die Termine pro Trainer zusammen, `HAVING COUNT(s.sessionid) > 15` filtert nach der Aggregation.
Ein Umweg über `studios` wäre falsch: Trainer «Elias Weber» (`TR030`) hat kein Studio (`studioid` NULL) und würde
bei einem `INNER JOIN` mit `studios` wegfallen, obwohl er 17 Termine geleitet hat.

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
SELECT
    t.firstname || ' ' || t.lastname AS trainer_name,
    COUNT(s.sessionid) AS session_count
FROM trainers t
JOIN sessions s ON t.trainerid = s.trainerid
GROUP BY t.trainerid, t.firstname, t.lastname
HAVING COUNT(s.sessionid) > 15
ORDER BY session_count DESC, trainer_name
""", con=engine)

engine.dispose()
df_C3

In [ ]:
# Kontrolle der Musterlösung
assert df_C3['trainer_name'].tolist() == ['Noah Gerber', 'Mia Wyss', 'Sophie Graf', 'Elias Weber']

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')